# MECG: Multilingual Evidence-Grounded Complaint Generation
## Notebook 08: Complete End-to-End Pipeline Demonstration

**Goal**: Demonstrate the unified 13-stage multimodal pipeline implemented in `src/pipeline.py`:
1. Video loading & keyframe extraction
2. 16 kHz audio extraction & Whisper transcription
3. Language detection & NLLB-200 translation
4. Qwen3-VL timestamped visual evidence extraction
5. DeBERTa-v3 aspect & emotion classification
6. SigLIP 2 + FAISS reference retrieval
7. Qwen3-VL evidence-grounded complaint generation
8. Factual claim decomposition & DeBERTa NLI verification
9. Evidence Grounding Score calculation and structured JSON reporting.

In [ ]:
# Pipeline Initialization
import os
import sys
import json
from pathlib import Path

CURRENT_DIR = Path.cwd()
if str(CURRENT_DIR) not in sys.path:
    sys.path.append(str(CURRENT_DIR))
if str(CURRENT_DIR.parent) not in sys.path:
    sys.path.append(str(CURRENT_DIR.parent))

from src.pipeline import MECGPipeline

pipeline = MECGPipeline()
print('MECGPipeline initialized and ready for multimodal inference!')

### 1. Run Pipeline on Sample Case
We run the complete pipeline on a test case with video and customer testimony.

In [ ]:
video_dir = Path('data/videos')
video_files = sorted(list(video_dir.glob('*.mp4')))

sample_video = video_files[0] if video_files else Path('data/sample_test.mp4')
sample_text = 'Plastic bottle smells strongly like chemicals, bottle cap is loose and water leaks out.'

print(f'Running pipeline on video: {sample_video.name}')
result = pipeline.process(
    video_path=sample_video,
    customer_text=sample_text,
    video_id='CASE_001',
    product_domain='Household Item',
    requested_output_language='en'
)

print('\nPipeline Execution Output JSON:')
print(json.dumps(result, indent=2, ensure_ascii=False))

### 2. Inspect Verified Claims and Grounding Score
Examine how the claim verification pipeline separated grounded facts from unsupported assertions.

In [ ]:
print(f"Evidence Grounding Score: {result['evidence_grounding_score']}")
print(f"Aspects Detected        : {result['aspects']}")
print(f"Emotion Detected        : {result['emotion']}")
print(f"\nVerified Claims ({len(result['verified_claims'])}):")
for vc in result['verified_claims']:
    print(f"  - Claim: '{vc['claim']}'")
    print(f"    Status: {vc['status']} | Source: {vc['evidence_source']}")

print(f"\nUnsupported Claims ({len(result['unsupported_claims'])}):")
for uc in result['unsupported_claims']:
    print(f"  - Claim: '{uc['claim']}' (No backing evidence found)")

### Summary & Next Steps
- End-to-end inference pipeline successfully demonstrated.
- Proceed to **Notebook 09: System Evaluation & Ablation Studies**.